# Modelos *Ensemble*

Una de las tendencias más recientes en el ámbito del modelado de inteligencia artificial puede resumirse bajo la expresión **“el conocimiento del conjunto o de la multitud”**. Esta formulación, relativamente familiar, hace referencia al empleo de una multiplicidad de modelos denominados *débiles* en el contexto de un meta-clasificador. El propósito de este enfoque es generar un modelo *fuerte* a partir del conocimiento extraído por dichos modelos *débiles*.  

Por ejemplo, aunque se detallará con mayor profundidad más adelante, en un **Random Forest** se desarrollan múltiples Árboles de Decisión de menor complejidad. La combinación de estos en el bosque aleatorio permite alcanzar un rendimiento superior al de cualquiera de los modelos individuales considerados por separado. Los modelos que se construyen de esta manera, ya sea en forma de meta-clasificadores o meta-regresores, se denominan de forma genérica **modelos *Ensemble***.  

Conviene destacar que estos modelos no se limitan exclusivamente al uso de árboles de decisión, sino que pueden estar conformados por cualquier tipo de modelo de aprendizaje automático previamente estudiado. Incluso es posible construir **modelos mixtos**, en los que no todos los submodelos se hayan obtenido mediante el mismo procedimiento, sino a través de la combinación de diversas técnicas, tales como *k*-NN, *SVM*, entre otras.  

De este modo, el **primer criterio para clasificar los modelos *Ensemble*** consiste en determinar si se trata de modelos **homogéneos** o **heterogéneos**. No obstante, este no es el único criterio de clasificación. En esta unidad se abordarán diferentes estrategias para la generación de modelos y para su combinación posterior. Asimismo, se analizarán en detalle dos de las técnicas más representativas dentro de los modelos *Ensemble*: **Random Forest** , **_XGBoost_** y otras de las últimas tendencias como **LightGBM** y **CatBoost**.  


En primer lugar, asegúrate de que los paquetes necesarios estén instalados. Por tanto, esta celda debe ejecutarse únicamente la **primera vez**.


In [ ]:
using Pkg
# Asegúrate de que los paquetes requeridos estén disponibles  
#(descomenta las líneas correspondientes para instalarlos, si es necesario).
Pkg.add([
    "MLJ", 
    "MLJBase", 
    "MLJModels", 
    "MLJEnsembles", 
    "MLJLinearModels", 
    "DecisionTree", 
    "MLJDecisionTreeInterface", 
    "NaiveBayes", 
    "EvoTrees", 
    "CategoricalArrays", 
    "Random",
    "LIBSVM",           
    "Plots",            
    "MLJModelInterface", 
    "CSV",              
    "DataFrames",       
    "UrlDownload",      
    "XGBoost"    
])

## Preparación de los datos

A diferencia de los primeros tutoriales, en los que se empleó el problema de las flores *iris* como referencia, en este tutorial utilizaremos un caso distinto. El problema también se encuentra disponible en el repositorio UCI. Aunque se trata igualmente de un conjunto de datos de pequeño tamaño, el número de variables aumenta de forma significativa, lo que nos permitirá realizar un análisis más amplio.  

En concreto, se trata de un problema clásico de aprendizaje automático conocido informalmente como **¿Roca o Mina?** (*Rock or Mine?*). Este conjunto de datos contiene 111 patrones correspondientes a rocas y 97 a minas submarinas (simuladas como cilindros metálicos). Cada patrón está formado por **60 medidas numéricas** que representan una sección de las secuencias de sonar. Estos valores se encuentran ya en el rango de 0.0 a 1.0, aunque resulta recomendable normalizarlos por seguridad. Las medidas reflejan el valor energético de diferentes rangos de longitud de onda durante un determinado periodo temporal.  

En este proceso vamos a emplear un par de paquetes nuevos, concretamente [DataFrames.jl](https://juliaai.github.io/DataScienceTutorials.jl/data/dataframe/) y [UrlDownload.jl](https://github.com/Arkoniak/UrlDownload.jl).  
Por tanto, lo primero que debemos hacer es asegurarnos de que los paquetes estén correctamente instalados.


In [ ]:
using Pkg;
Pkg.add("CSV")
Pkg.add("DataFrames")
Pkg.add("UrlDownload")

A continuación, los datos se descargarán en caso de que no estén disponibles localmente.  
Para ello, puede emplearse el siguiente fragmento de código:

In [ ]:
using UrlDownload
using DataFrames
using CSV
using CategoricalArrays

url = "https://archive.ics.uci.edu/ml/machine-learning-databases/undocumented/connectionist-bench/sonar/sonar.all-data"
data = urldownload(url, true, format=:CSV, header=false) |> DataFrame
describe(data)

Como puede observarse en la línea anterior, hemos descargado los datos y los hemos canalizado mediante el operador `|>` hacia la función `DataFrame`.  
Esto permite crear una estructura similar a una tabla de base de datos, lo cual resulta especialmente conveniente para comprobar la existencia de valores ausentes o analizar los rangos de las distintas variables.  

De hecho, esta biblioteca facilita notablemente la gestión de valores perdidos, al proporcionar funciones que permiten tanto completar como eliminar las muestras con medidas no válidas.  
No obstante, el conjunto de datos es demasiado extenso para visualizar todas las variables en el informe de salida. Si realizamos algunas consultas específicas, podemos comprobar que **no existen valores ausentes**. Además, ninguna variable supera el valor de 1.0, aunque algunas de ellas no están completamente normalizadas.  

Una estructura análoga puede encontrarse en otros lenguajes de programación, como **R** o **Python**.  

Como ejemplo de este proceso, añadiremos una columna adicional con el fin de **convertir en categórica la última columna (la número 60)**, que contiene una **M** para cada muestra correspondiente a una mina y una **R** para cada muestra correspondiente a una roca.


In [ ]:
insertcols!(data, :Mine => data[:, 61].=="M")

Una vez que los datos se han cargado en el `DataFrame` para su verificación y se ha aplicado cualquier posible proceso de preparación, es necesario transformarlos para su utilización en los modelos.  

Al igual que en los tutoriales anteriores, los datos deben convertirse a una **forma matricial**, tal como se muestra a continuación:


In [ ]:
input_data = Matrix{Float64}(data[!, 1:60]);
output_data = categorical(data[!, :Mine]);

@assert input_data isa Matrix
@assert output_data isa CategoricalArray

Cabe destacar que, en un `DataFrame`, cuando se consulta un conjunto de filas, como en el caso de la variable `X`, el resultado obtenido es también un objeto de tipo `DataFrame`.  

Por tanto, para poder aplicar las operaciones posteriores, es necesario utilizar la función `Matrix`, la cual convierte dicho resultado en una **matriz** sobre la que pueden ejecutarse las operaciones habituales.


### Pregunta

> ❓Ahora, los datos se han cargado y convertido a los tipos habituales. A continuación, deberías poder abordar la siguiente sección: realizar una partición del conjunto de datos en dos subconjuntos, **entrenamiento** y **prueba**, y aplicar la normalización correspondiente. Incluye el código en la sección siguiente para ejecutar ambas operaciones. *Consejo: debido a la preparación para los modelos de MLJ, lee las notas al final del documento.*


In [ ]:
#train_input, train_output, test_input, test_output = #TODO


## Línea base

Como se ha mencionado, los *ensembles* son conjuntos de clasificadores “más débiles” que, al combinarse, permiten superar sus limitaciones individuales. Por ello, antes de abordar los *ensembles*, resulta necesario disponer de algunos **modelos de referencia** que posteriormente se integrarán en un **meta-clasificador**.

En el ejemplo siguiente, se entrenan varios modelos sencillos implementados con la biblioteca `MLJ`:  
- una **SVM** con **núcleo RBF**  
- una **Regresión Lineal**  
- un **Naïve Bayes**  
- un **Árbol de Decisión**.

In [ ]:

using MLJ
using MLJBase: accuracy

# Cargar los modelos (tenga presente que si algunos no están instalados, MLJ le pedirá que los instale)
SVC = @load ProbabilisticSVC pkg=LIBSVM
LogisticClassifier = @load LogisticClassifier pkg=MLJLinearModels
DecisionTreeClassifier = @load DecisionTreeClassifier pkg=DecisionTree
GaussianNBClassifier = @load GaussianNBClassifier pkg=NaiveBayes

#Definir los modelos base
models = Dict(
    "SVM" => SVC(),
    "LR"  => LogisticClassifier(),
    "DT"  => DecisionTreeClassifier(max_depth=4),
    "NB"  => GaussianNBClassifier(),
)

base_models=  [ model for (name, model) in models]

machines = Dict()

In [ ]:
# Ejecutar el entrenamiento de cada modelo y calculas los valores de test (accuracy)
for (name, model) in models
    machines[name] = machine(model, train_input, train_output) |> fit!
    acc = MLJ.accuracy(predict_mode(machines[name], test_input), test_output)
    println("$name: $(acc*100) %")
end

## Combinación de modelos débiles en un *ensemble*

A la hora de combinar los modelos, existen diversas estrategias que dependen de la tarea que se esté abordando, es decir, de si se trata de un problema de **clasificación** o de **regresión**. En este caso concreto nos centraremos en la clasificación; no obstante, para la regresión el procedimiento sería similar, aunque habría que tener en cuenta la naturaleza continua de los valores al combinar las salidas.

En lo que respecta a la **combinación de clasificadores**, existen principalmente dos enfoques para integrar las salidas de varios modelos:  
- **Votación mayoritaria** (*Majority Voting*), y  
- **Votación mayoritaria ponderada**, también conocida como **Votación Suave** (*Soft Voting*).


Como primer paso y dado que MLJ no dispone de una clase para este tipo de clasificaciones de manera nativa, vamos a definir una que soporte modelos heterogéneos

In [ ]:
using MLJ
using MLJBase
using MLJModelInterface
using CategoricalArrays

# ===================================================
# DEFINICION del VOTINGCLASSIFIER compatible con MLJ
# ===================================================

"""
    VotingClassifier <: Probabilistic

Un clasificador *ensemble* que combina las predicciones de múltiples modelos base utilizando diferentes estrategias de votación.

# Campos
- `models::Vector{Probabilistic}`: Vector de modelos probabilísticos base que se combinarán.  
- `voting::Symbol`: Estrategia de votación, que puede ser `:hard` (votación mayoritaria) o `:soft` (promedio de probabilidades).  
- `weights::Union{Nothing, Vector{Float64}}`: Pesos opcionales para cada modelo. Si se establece como `nothing`, todos los modelos tendrán el mismo peso. Los pesos se normalizan automáticamente para que su suma sea 1.0.

# Ejemplos
```julia
# Pesos iguales (por defecto)
voting_clf = VotingClassifier(
    models=[LogisticClassifier(), DecisionTreeClassifier()],
    voting=:soft
)

# Pesos personalizados (se normalizan automáticamente)
voting_clf = VotingClassifier(
    models=[LogisticClassifier(), DecisionTreeClassifier(), RandomForestClassifier()],
    voting=:hard,
    weights=[5, 3, 2]  # Se normalizarán a [0.5, 0.3, 0.2]
)
```
"""
mutable struct VotingClassifier <: Probabilistic   # Models must be probabilistic, inherited from MLJBase
    models::Vector{Probabilistic}
    voting::Symbol  # :hard or :soft
    weights::Union{Nothing, Vector{Float64}}
end


"""
    VotingClassifier(; models=Probabilistic[], voting=:hard, weights=nothing)
Constructor del `VotingClassifier`.

# Argumentos
- `models::Vector{Probabilistic}=Probabilistic[]`: Modelos base que se combinarán.  
- `voting::Symbol=:hard`: Estrategia de votación (`:hard` o `:soft`).  
- `weights::Union{Nothing, Vector{<:Real}}=nothing`: Pesos asignados a cada modelo. Se normalizan automáticamente para que su suma sea 1.0.

# Excepciones
- `AssertionError`: Si el parámetro `voting` no es `:hard` ni `:soft`.  
- `AssertionError`: Si la longitud del vector de pesos no coincide con el número de modelos.  
- `AssertionError`: Si todos los pesos son cero o negativos.
"""

function VotingClassifier(; models=Probabilistic[], voting=:hard, weights=nothing)
    @assert voting in [:hard, :soft] "The only possible labels are :hard or :soft"
    
    normalized_weights = nothing
    if weights !== nothing
        @assert length(weights) == length(models) "El número de pesos tiene que ser igual al de modelos"
        @assert all(w >= 0 for w in weights) "Todos los pesos deben de ser no negativos"
        @assert sum(weights) > 0 "Al menos uno de los pesos debe ser positivo"
        
        # Suma normalizada de los pesos 1.0
        normalized_weights = Float64.(weights) ./ sum(weights)
    end
    
    return VotingClassifier(models, voting, normalized_weights)
end

"""
    MLJModelInterface.fit(model::VotingClassifier, verbosity::Int, X, y)

Entrena el `VotingClassifier` ajustando cada modelo base con los datos proporcionados.

# Argumentos
- `model::VotingClassifier`: Instancia del clasificador de votación.  
- `verbosity::Int`: Nivel de verbosidad para el registro del proceso de entrenamiento.  
- `X`: Características de entrenamiento (en formato de tabla).  
- `y`: Variable objetivo de entrenamiento (vector categórico).

# Retorna
- `fitresults`: Vector de máquinas entrenadas (una por cada modelo base).  
- `cache`: `nothing` (no se implementa almacenamiento en caché).  
- `report`: Tupla con nombre que contiene información del entrenamiento (número de modelos, estrategia de votación y pesos normalizados).
"""

function MLJModelInterface.fit(model::VotingClassifier, verbosity::Int, X, y)
    # Entrenar cada modelos base
    machs = [begin
        mm = machine(deepcopy(m), X, y)
        fit!(mm, verbosity=0)
        mm
    end for m in model.models]

    fitresults = (
        machines = machs,
        class_levels = collect(levels(y)),   #Mantener las clases para futuras ejecuciones
        class_pool = CategoricalArrays.pool(y)
    )
    
    # Guardar la información necesaria
    cache = nothing
    report = (n_models=length(model.models), voting=model.voting, weights=model.weights)
    
    return fitresults, cache, report
end


"""
    MLJModelInterface.predict_mode(model::VotingClassifier, fitresult, Xnew)

Predice las etiquetas de clase utilizando **votación dura** (*hard voting*), es decir, votación mayoritaria con pesos opcionales.

# Argumentos
- `model::VotingClassifier`: Instancia del clasificador de votación.  
- `fitresult`: Vector de máquinas entrenadas obtenido en la fase de ajuste.  
- `Xnew`: Nuevos datos sobre los que se realizará la predicción.

# Retorna
- Vector categórico con las etiquetas de clase predichas, calculadas mediante votación mayoritaria (ponderada o no).

# Detalles
Cada modelo base emite un voto por una clase.  
Si se han definido pesos, cada voto se multiplica por el peso correspondiente.  
La clase con el mayor número de votos (ponderados) es seleccionada como predicción final.
"""

function MLJModelInterface.predict_mode(model::VotingClassifier, fitresult, Xnew)
    machines = fitresult.machines
    class_levels = fitresult.class_levels
    
    # Obtener las prediciones de todos los modelos
    predictions = [categorical(predict_mode(mach, Xnew), levels=class_levels) for mach in machines]
    
    # Recoger datos básicos de la simulación
    n_samples = length(predictions[1])
    n_models = length(machines)
    
    # Establecer todos los pesos iguales, si no se han especificado
    weights = model.weights === nothing ? fill(1.0/n_models, n_models) : model.weights
    
    # Output Vector with the same type as pthe predictions
    ensemble_pred = similar(predictions[1])
    
    for i in 1:n_samples
        # Acumular votos en el orden de class_levels; los empates conservan la primera clase.
        vote_scores = zeros(length(class_levels))

        for (j, prediction) in enumerate(predictions)
            predicted_class = prediction[i]
            class_index = findfirst(class_label -> class_label == predicted_class, class_levels)
            @assert class_index !== nothing "La clase predicha no se encuentra entre las clases de entrenamiento"
            vote_scores[class_index] += weights[j]
        end

        ensemble_pred[i] = class_levels[argmax(vote_scores)]
    end

    return ensemble_pred
end

"""
    MLJModelInterface.predict(model::VotingClassifier, fitresult, Xnew)

Predice las probabilidades de clase utilizando la estrategia de votación especificada.

# Argumentos
- `model::VotingClassifier`: Instancia del clasificador de votación.  
- `fitresult`: Vector de máquinas entrenadas obtenido durante el ajuste.  
- `Xnew`: Nuevos datos sobre los que se realizarán las predicciones.

# Retorna
- Vector de distribuciones `UnivariateFinite` que representan las probabilidades de pertenencia a cada clase.

# Detalles
- Para la votación `:hard`: se devuelven predicciones deterministas encapsuladas en `UnivariateFinite` (con pesos opcionales).  
- Para la votación `:soft`: se calculan las probabilidades promediando las distribuciones generadas por todos los modelos base, aplicando los pesos correspondientes.
"""

function MLJModelInterface.predict(model::VotingClassifier, fitresult, Xnew)
    machines     = fitresult.machines
    class_levels = fitresult.class_levels
    class_pool   = fitresult.class_pool

    result = if model.voting == :hard
       # Hard Voting
        yhat = MLJModelInterface.predict_mode(model, fitresult, Xnew)
        yhat = categorical(yhat; levels=class_levels)  # asegura mismos niveles

        # Devuelve las probabilidades como one-hot encoded 
        [MLJBase.UnivariateFinite(
                    class_levels,
                    [lvl == yhat[i] ? 1.0 : 0.0 for lvl in class_levels];
                    pool=class_pool
                ) for i in 1:length(yhat)]
    else
        # Soft voting
        all_predictions = [predict(mach, Xnew) for mach in machines]

        n_samples = length(all_predictions[1])
        n_models  = length(machines)
        n_classes = length(class_levels)
        weights   = model.weights === nothing ? fill(1.0/n_models, n_models) : model.weights

        avg_probs = zeros(n_samples, n_classes)
        for (w, prediction) in zip(weights, all_predictions)
            for i in 1:n_samples
                p_i = prediction[i]
                if p_i isa MLJBase.UnivariateFinite
                    for (j, level) in enumerate(class_levels)
                        avg_probs[i, j] += w * pdf(p_i, level)
                    end
                else
                    # determinista -> one-hot
                    for (j, level) in enumerate(class_levels)
                        avg_probs[i, j] += w * (p_i == level ? 1.0 : 0.0)
                    end
                end
            end
        end

        # Normalizar cada probabilidad para evitar problemas con el redondeo con los números reales
        for i in 1:n_samples
            s = sum(@view avg_probs[i, :])
            if s > 0
                @. avg_probs[i, :] = avg_probs[i, :] / s
            end
        end

        # Usa la misma codificación entre llamadas para prevenir confusiones
        [MLJBase.UnivariateFinite(class_levels, collect(@view avg_probs[i, :]); pool=class_pool)
         for i in 1:n_samples]
    end

    return result
end
"""
Registro de metadatos del modelo para `VotingClassifier`.

Especifica los tipos de entrada/salida y las capacidades para su integración con MLJ.
"""
MLJModelInterface.metadata_model(VotingClassifier,
    input_scitype=Table(Continuous),
    target_scitype=AbstractVector{<:Finite},
    supports_weights=false,
    load_path="VotingClassifier"
)

## Votación mayoritaria

También conocida como *Hard Voting*, consiste, como su nombre sugiere, en seleccionar la opción más votada entre las predicciones emitidas por los distintos modelos. Cada modelo emite un voto o predicción determinista. La clase (o predicción) final es aquella que recibe **el mayor número de votos** o, en caso de regresión, **el valor medio** entre los resultados. Es equivalente a una “elección democrática” en la que cada modelo/experto dispone de un voto y gana la opción más votada. De este modo, el problema puede abordarse teniendo en cuenta diferentes resultados o puntos de vista.

### Ejemplo

Con 3 clasificadores que predicen un mismo patrón:

- **SVM** predice: **Mina**  
- **Regresión Logística** predice: **Roca**  
- **Naive Bayes** predice: **Roca**

**Resultado:** **Roca** (2 votos frente a 1) ✓

A continuación se muestra un ejemplo en código de cómo construir un modelo de este tipo.


In [ ]:
#Define el **meta-clasificador** a partir de los `base_models`.

models["Ensemble (Hard Voting)"] = VotingClassifier(models = base_models, voting=:hard)
machines["Ensemble (Hard Voting)"] = machine(models["Ensemble (Hard Voting)"], train_input, train_output) |> fit!

for (name, machine) in machines
    acc = MLJ.accuracy(MLJ.mode.(predict(machine, test_input)), test_output)
    println("$name: $(acc*100) %")
end

El problema principal radica en que **confiamos por igual en todos los modelos** a la hora de decidir la clase de respuesta.

## *Soft Voting* 

Como se mencionó en la sección anterior, uno de los problemas de los modelos *ensemble* clásicos es que **todos los resultados se ponderan de igual manera** y que, en cada uno de los modelos “débiles”, solo se tiene en cuenta la clase más votada.  

Para resolver esta limitación, el *Soft Voting* propone utilizar las **probabilidades** que cada clasificador asigna a cada clase, en lugar de considerar únicamente la clase predicha. El resultado final se obtiene promediando dichas probabilidades y seleccionando la clase con la **mayor probabilidad promedio**.

### Ejemplo sin pesos (todos los modelos igualmente importantes)

| Clasificador | P(Mina) | P(Roca) |
|---------------|---------|---------|
| SVM           | 0.9     | 0.1     |
| Regresión Logística | 0.3 | 0.7 |
| Naive Bayes   | 0.2     | 0.8 |
| **Promedio**  | **0.47**| **0.53** |

**Cálculo:**
- P(Mina) = (0.9 + 0.3 + 0.2) / 3 = 0.47  
- P(Roca) = (0.1 + 0.7 + 0.8) / 3 = 0.53  

**Resultado:** **Roca** (mayor probabilidad promedio) ✓  

**Ventaja frente al _Hard Voting_:**  
Aunque el modelo SVM muestra una alta confianza en la clase *Mina* (0.9), los otros dos modelos manifiestan una confianza significativa en *Roca* (0.7 y 0.8). La votación suave permite **capturar y aprovechar esta información de confianza** en la combinación final.


In [ ]:
#Define el **meta-clasificador** a partir de los `base_models`.

models["Ensemble (Soft Voting - Equal)"] = VotingClassifier( models = base_models, voting = :soft, weights = nothing) # All models equally weighted
machines["Ensemble (Soft Voting - Equal)"] = machine(models["Ensemble (Soft Voting - Equal)"], train_input, train_output) |> fit!

for (name, machine) in machines
    acc = MLJ.accuracy(MLJ.mode.(predict(machine, test_input)), test_output)
    println("$name: $(acc*100) %")
end

#### *Weighted Soft Voting* (Votación ponderada)

Aunque *Soft Voting* mejora el proceso al tener en cuenta el grado de confianza de cada modelo, el principal inconveniente sigue siendo la **igualdad de influencia entre los modelos**.  
Para solucionar este aspecto, una de las propuestas consiste en **introducir un sistema de ponderación en la decisión**.  

En muchos casos sabemos que algunos modelos presentan un mejor rendimiento que otros.  
Por ejemplo, si una **SVM** alcanza una precisión del **85 %**, mientras que los demás modelos rondan el **70 %**, es razonable asignar **mayor peso** a la SVM.  
Esto se consigue mediante el uso de **pesos** en la votación suave, donde los valores de probabilidad de cada modelo se **multiplican por su peso** antes de calcular el promedio.  

Matemáticamente:

$$
P(\text{clase}) = \frac{\sum_{i=1}^{n} w_i \cdot P_i(\text{clase})}{\sum_{i=1}^{n} w_i}
$$

Donde:

* $w_i$ = peso asignado al modelo $i$  
* $P_i(\text{clase})$ = probabilidad asignada por el modelo $i$ a la clase  
* $n$ = número total de modelos  

Siguiendo el mismo ejemplo anterior, supongamos que deseamos **incrementar la importancia de la SVM**.

### Ejemplo con pesos [2, 1, 1] (peso doble para la SVM)

| Clasificador | Peso | P(Mina) | P(Roca) | Contribución Mina | Contribución Roca |
|---------------|------|----------|----------|-------------------|-------------------|
| SVM           | 2    | 0.9      | 0.1      | 2 × 0.9 = 1.8     | 2 × 0.1 = 0.2     |
| Reg. Logística| 1    | 0.3      | 0.7      | 1 × 0.3 = 0.3     | 1 × 0.7 = 0.7     |
| Naive Bayes   | 1    | 0.2      | 0.8      | 1 × 0.2 = 0.2     | 1 × 0.8 = 0.8     |
| **Suma**      | 4    |          |          | **2.3**           | **1.7**           |
| **Promedio ponderado** |  |    |          | **0.575**         | **0.425**         |

**Cálculo:**
- P(Mina) = (1.8 + 0.3 + 0.2) / 4 = 2.3 / 4 = 0.575  
- P(Roca) = (0.2 + 0.7 + 0.8) / 4 = 1.7 / 4 = 0.425  

**Resultado:** **Mina** (mayor probabilidad ponderada) ✓


In [ ]:
models["Ensemble (Soft Voting - Weighted)"] = VotingClassifier(models = base_models, voting=:soft,weights=[1,2,2,1])
machines["Ensemble (Soft Voting - Weighted)"] = machine(models["Ensemble (Soft Voting - Weighted)"], train_input, train_output) |> fit!

for (name, machine) in machines
    acc = MLJ.accuracy(MLJ.mode.(predict(machine, test_input)), test_output)
    println("$name: $(acc*100) %")
end

## Cuándo utilizar cada estrategia

### *Hard Voting*

- Cuando los modelos solo generan predicciones categóricas (sin probabilidades).  
- Cuando todos los modelos tienen una fiabilidad similar.  
- Es un método más **simple y rápido**.

### *Soft Voting*

- Cuando los modelos proporcionan **probabilidades** como salida.  
- Cuando todos los modelos presentan **rendimientos comparables**.  
- Permite **capturar el grado de confianza** en cada predicción.

### *Weighted Soft Voting*

- Cuando algunos modelos son claramente **más precisos o robustos** que otros.  
- Cuando se desea **asignar mayor importancia** a determinados modelos.  
- Los pesos pueden definirse en función de:
    - La **precisión de validación**.  
    - La **especialización o experiencia** del modelo en un dominio concreto.  
    - Métricas relevantes como el **F1-score** u otras medidas de rendimiento.  

Para la **selección de los pesos**, pueden emplearse varias estrategias, siendo las más relevantes:

1. **Asignación manual**, basada en conocimiento previo del rendimiento de los modelos.  
2. **Basada en la precisión de validación**, ajustando los pesos en proporción a los resultados obtenidos.  
3. **Optimización mediante búsqueda en _Grid search_**), explorando distintas combinaciones de pesos para maximizar el rendimiento global del *ensemble*.


### Pregunta
> ❓Hemos realizado todas las pruebas utilizando una **estrategia *hold-out***; sin embargo, como se indicó en una sesión anterior, se prefiere la aplicación de un enfoque de **validación cruzada (*cross-validation*)** para reducir la dependencia de la selección de las muestras.  

En este caso, pueden considerarse dos enfoques diferentes:  

1. Aplicar la validación cruzada a cada modelo individualmente, elegir el mejor y combinarlos en un único *ensemble*.  
2. Aplicar la validación cruzada al nivel del *ensemble* antes de entrenar los modelos.  

¿Cuál de ellos es el correcto y por qué?

`Answer here`

### *Stacking*

Este último enfoque para la combinación de modelos puede considerarse una __variante *Soft Voting*__.   
Como se mencionó en dicha sección, la votación suave permite fijar los pesos de cada uno de los modelos, y estos pueden ajustarse mediante una técnica de **gradiente descendente**.  

El *Stacking* suele identificarse como una técnica de clasificación superior a una regresión lineal (que es, en esencia, lo que realiza la votación suave), al emplear un **modelo más complejo**, como una **red neuronal artificial (ANN)**, para combinar los resultados de los modelos base.  

Así, de manera análoga a lo realizado anteriormente, las salidas de las diferentes técnicas pueden tomarse y utilizarse como **entradas de otro modelo de clasificación**, lo que permite ajustar los pesos y considerar **combinaciones no lineales** de las respuestas de cada modelo.  

A continuación se muestra un ejemplo de este proceso en código, utilizando la implementación de `MLJ`, que emplea un **SVC como modelo combinador**:


In [ ]:
# Crea una `NamedTuple` solo con los modelos base originales.
base_model_names = ["SVM", "LR", "DT", "NB"]
stack_base_models = (; (Symbol(name) => models[name] for name in base_model_names)...)

# Construir el stacking:
# - resampling=CV(...) define cómo se generan las predicciones out-of-fold
# - measures=... solo para reporte interno; no afecta al entrenamiento final del stack
models["Ensemble (Stacking)"] = Stack(; 
    metalearner = SVC(),
    resampling = CV(nfolds=5, shuffle=true, rng=123),
    measures = log_loss,
    stack_base_models...  # Expande la `NamedTuple` de modelos base.
)

# Entrenar el stack en el train dataset
machines["Ensemble (Stacking)"] = machine(models["Ensemble (Stacking)"], train_input, train_output) |> fit!

In [ ]:
for (name, machine) in machines
    acc = MLJ.accuracy(MLJ.mode.(predict(machine, test_input)), test_output)
    println("$name: $(acc*100) %")
end

## Creación de modelos

Uno de los elementos clave que aún no se ha abordado es la creación de los modelos que compondrán el meta-clasificador. Hasta ahora, el enfoque seguido no es muy adecuado, ya que el conjunto de datos de entrada para todos los modelos es el mismo. Esto provoca una evidente falta de diversidad en los modelos, dado que, sea cual sea el modelo que creemos, dispondrá de la misma información o “punto de vista” que los demás. Sin embargo, esta no es la práctica habitual. En su lugar, el conjunto de patrones de entrada suele **dividirse en subconjuntos más pequeños** con los que entrenar una o varias técnicas, con el doble objetivo de **reducir el coste computacional** por un lado y **aumentar la diversidad de los modelos** por otro. Conviene recordar en este punto que los modelos “débiles” no tienen por qué ser perfectos en todas las clases ni cubrir todas las posibilidades; basta con que sean **rápidos de entrenar** y ofrezcan una salida **razonablemente consistente**.

En cuanto a la forma de particionar los datos para la creación de los modelos, la mayoría de los enfoques suelen considerar dos estrategias principales, conocidas como **_Bagging_** y **_Boosting_**. A continuación se describen brevemente ambas.

### Bagging o agregación *bootstrap*
La técnica conocida como _Bagging_ o selección con reemplazo fue propuesta por **Breitman** en 1996. Se basa en el desarrollo de múltiples modelos que pueden entrenarse en **paralelo**. El elemento clave de estos modelos es que **cada modelo se entrena sobre un subconjunto del conjunto de entrenamiento**. Este subconjunto de datos se extrae **aleatoriamente con reemplazo**. Este último punto es especialmente importante porque, una vez que se ha seleccionado un ejemplo de entre las posibilidades, se **vuelve a colocar** entre las posibilidades para que pueda ser seleccionado nuevamente, ya sea en el subconjunto que se está construyendo o en los subconjuntos de otros modelos; es decir, se crean **conjuntos de ejemplos no disjuntos**.

![Bagging Example](https://upload.wikimedia.org/wikipedia/commons/thumb/c/c8/Ensemble_Bagging.svg/440px-Ensemble_Bagging.svg.png)

El resultado es la creación de “expertos” en datos **especializados** y dependientes de la partición. Si bien los datos **comunes** o más frecuentes quedan correctamente cubiertos por todos los modelos, también es cierto que los datos **menos frecuentes** tienden a no estar presentes en todas las particiones y pueden no quedar cubiertos en todos los casos. De este modo, se obtienen modelos **más especializados** en ciertos datos o con un **punto de vista distinto**, que se comportan como expertos en una región particular del espacio de búsqueda.

Aunque se tratará con mayor detalle más adelante, una técnica bien conocida que emplea este enfoque para la construcción de sus modelos “débiles” es **RandomForest**. En efecto, construye los árboles de decisión que componen el meta-clasificador de esta manera. **Cualquier clasificador** puede utilizarse como base de un _Bagging_ mediante la clase [EnsembleModel](https://juliaai.github.io/MLJ.jl/stable/models/EnsembleModel_MLJEnsembles/#EnsembleModel_MLJEnsembles).

Por ejemplo, en el siguiente código se han elegido **10 SVM para clasificación** como modelos débiles. Cada uno de estos modelos se ha entrenado **solo con el 50 %** de los patrones de entrenamiento, por lo que debería **incrementarse la varianza** entre ellos.

 

In [ ]:
# Añade un modelo de *Bagging* utilizando **SVC** como modelo base.
using MLJEnsembles: EnsembleModel, CPUThreads

models["Bagging (SVC)"] = EnsembleModel(
    model = SVC(),              # o ProbabilisticSVC()
    n = 10,                     # numbero de modelos base 
    bagging_fraction = 0.50,    # fraction de eejemplos por modelo base
    rng = 123,                  
    acceleration = CPUThreads() # Utiliza `Threads` para acelerar el entrenamiento, dada la independencia de los modelos base.
)

machines["Bagging (SVC)"] = machine(models["Bagging (SVC)"], train_input, train_output) |> fit!

for (name, machine) in machines
    acc = MLJ.accuracy(MLJ.mode.(predict(machine, test_input)), test_output)
    println("$name: $(acc*100) %")
end

El parámetro `bagging_fraction` controla la fracción de ejemplos de entrenamiento (filas) utilizada para entrenar cada modelo base. Una partición vertical del conjunto de datos, es decir, seleccionar solo un subconjunto de características (columnas), constituye una fuente distinta de diversidad: depende del modelo base o de algoritmos *ensemble* que la soporten explícitamente, como Random Forest mediante parámetros como `n_subfeatures`. Este enfoque resulta útil cuando el número de características es elevado, porque cada modelo débil puede centrarse en una vista diferente del espacio de entrada.


### *Boosting*

La otra gran familia de técnicas para el meta-modelado *ensemble* es la conocida como **_Boosting_**.En este caso, el enfoque es ligeramente diferente, ya que el objetivo es crear una **cadena de clasificadores**.La idea clave es que cada clasificador sucesivo se vuelve **más especializado en los patrones que los modelos anteriores clasificaron incorrectamente**.  

Así, al igual que en el caso anterior, se selecciona un subconjunto de patrones del conjunto original, pero este proceso se realiza **de manera secuencial y sin reemplazo**.Esto provoca que los nuevos aprendices se centren progresivamente en los casos más difíciles, generando gradualmente un modelo compuesto más robusto y preciso.  

Por tanto, como en el *Bagging*, la idea subyacente de este enfoque es que **no todos los modelos deben utilizar todos los patrones como base**, pero, a diferencia del *Bagging*, este proceso es **lineal**, debido a la **dependencia entre la construcción de los modelos**. Finalmente, las salidas de los modelos individuales se combinan mediante una **votación mayoritaria ponderada**, donde el peso de cada clasificador refleja su rendimiento durante el entrenamiento.

<img src="https://upload.wikimedia.org/wikipedia/commons/thumb/b/b5/Ensemble_Boosting.svg/1920px-Ensemble_Boosting.svg.png" alt="Boosting examples" width="600"/>

#### AdaBoost

El algoritmo **AdaBoost** comienza asignando **pesos iguales** a todas las instancias del conjunto de entrenamiento. A continuación, se entrena un clasificador sencillo (denominado *stump*, que consiste en un árbol de decisión de un solo nivel) y se evalúa su rendimiento. Las instancias mal clasificadas reciben **mayor peso**, de modo que el siguiente clasificador se centra más en esos casos difíciles.  

Este proceso iterativo continúa, actualizando los pesos en cada iteración y creando un nuevo modelo débil que complementa a los anteriores. En **AdaBoost**, la ponderación tanto de las instancias como de los clasificadores se basa en una **función de pérdida exponencial**, que penaliza las clasificaciones erróneas de manera exponencial. La predicción final del *ensemble* se obtiene mediante una **votación mayoritaria ponderada** entre todos los clasificadores débiles.  

En **MLJ**, este comportamiento está implementado mediante el modelo `AdaBoostStumpClassifier`, proporcionado por el paquete `DecisionTree.jl`.

#### Gradient Boosting

El **Gradient Boosting** sigue un principio distinto: en lugar de reajustar los pesos de las instancias de forma explícita, utiliza un enfoque de **descenso por gradiente** para minimizar una función de pérdida. Cada nuevo árbol de la secuencia se entrena para predecir los **errores residuales** (o gradientes) del *ensemble* anterior, refinando gradualmente el modelo.  

En el caso de la clasificación, cada árbol de decisión modela la **verosimilitud logística** de los datos, y sus predicciones se combinan para estimar las probabilidades de clase. La decisión final se obtiene a partir de la **suma de estas probabilidades** a lo largo de todos los árboles.  

En **MLJ**, este procedimiento puede implementarse mediante el modelo `EvoTreeClassifier` (del paquete `EvoTrees.jl`), que es conceptualmente similar al `GradientBoostingClassifier` de *scikit-learn*, aunque está escrito íntegramente en Julia y admite **aceleración tanto por CPU como por GPU**.


In [ ]:
using MLJ
using MLJBase: accuracy

# Carga modelos (puro Julia)
AdaBoostStumpClassifier = @load AdaBoostStumpClassifier pkg=DecisionTree
EvoTreeClassifier = @load EvoTreeClassifier pkg=EvoTrees

# AdaBoost (similar a sklearn AdaBoostClassifier con stumps)
models["AdaBoost"] = AdaBoostStumpClassifier(n_iter = 30)
machines["AdaBoost"] = machine(models["AdaBoost"], train_input, train_output) |> fit!


# Gradient Boosting (similar a sklearn GradientBoostingClassifier)
models["EvoTrees"] = EvoTreeClassifier(
    nrounds=30,
    eta=1.0,
    max_depth=2,
    loss=:logistic
)

machines["EvoTrees"] = machine(models["EvoTrees"], train_input, train_output) |> fit!


for (name, machine) in machines
    acc = MLJ.accuracy(MLJ.mode.(predict(machine, test_input)), test_output)
    println("$name: $(acc*100) %")
end


### Pregunta
Desarrolla una función para **entrenar y evaluar un *ensemble* homogéneo** utilizando **`EnsembleModel`** de **MLJ**, donde todos los estimadores base sean del **mismo tipo**.  
La función, denominada `trainClassEnsemble`, debe seguir un esquema de **validación cruzada estratificada** y devolver, al menos, el valor de la métrica o métricas en test.

**Resumen de requisitos:**

1. Crear un vector de resultados de longitud *k* para almacenar la métrica de test en cada fold.  
2. Para cada fold, construir las particiones `X_train`, `y_train`, `X_test` e `y_test` usando los índices estratificados.  
3. Instanciar el modelo base con sus hiperparámetros.  
4. Construir el *ensemble* homogéneo envolviendo el modelo base en `EnsembleModel`, configurando `n`, `bagging_fraction`, `sampling_fraction`, `n_subfeatures` u otros parámetros aplicables.  
5. Entrenar el *ensemble* con `machine` y `fit!`.  
6. Si se requiere validación interna, aplicar un *hold-out* sobre `X_train` e `y_train`.  
7. Calcular la métrica de test del fold y almacenarla.  
8. Reportar el promedio y la desviación estándar de cada métrica.

> **Importante:** debes implementarlo como un *ensemble* homogéneo usando `EnsembleModel` de MLJ. No se requiere *stacking* ni *boosting* en esta pregunta.

> **Nota:** para mantener compatibilidad con unidades anteriores, la salida deseada se espera como una matriz booleana *one-hot*. Como MLJ espera etiquetas de clase, convierte la matriz *one-hot* a etiquetas categóricas antes de entrenar. Si usas las funciones de métricas previas, convierte las predicciones de MLJ de nuevo a formato *one-hot*.

In [ ]:
function trainClassEnsemble(estimator::Symbol,
        modelsHyperParameters::Dict,
        ensembleHyperParameters::Dict,
        trainingDataset::Tuple{AbstractArray{<:Real,2}, AbstractArray{Bool,2}},
        kFoldIndices::Array{Int64,1})

    inputs, targets_onehot = trainingDataset

    # Convertir las salidas one-hot de unidades previas a etiquetas categóricas para MLJ.
    classes = collect(1:size(targets_onehot, 2))
    targets = categorical([classes[argmax(row)] for row in eachrow(targets_onehot)])

    #TODO

    # Si se usan las métricas previas, convertir las predicciones MLJ a one-hot.
    # predicted_onehot = oneHotEncoding(predicted_labels, classes)
end

### Pregunta
> ❓ Repite la función anterior, pero esta vez para un clasificador **heterogéneo**.
>
> Usa la misma convención de entrada/salida y las mismas conversiones descritas en la pregunta anterior.


In [ ]:
function trainClassEnsemble(estimators::AbstractVector{Symbol},
        modelsHyperParameters::AbstractVector{<:Dict},
        ensembleHyperParameters::Dict,
        trainingDataset::Tuple{AbstractArray{<:Real,2}, AbstractArray{Bool,2}},
        kFoldIndices::Array{Int64,1})

    inputs, targets_onehot = trainingDataset

    # Convertir las salidas one-hot de unidades previas a etiquetas categóricas para MLJ.
    classes = collect(1:size(targets_onehot, 2))
    targets = categorical([classes[argmax(row)] for row in eachrow(targets_onehot)])

    #TODO

    # Si se usan las métricas previas, convertir las predicciones MLJ a one-hot.
    # predicted_onehot = oneHotEncoding(predicted_labels, classes)
end

## Técnicas que integran el enfoque *Ensemble*

Algunos de los algoritmos más conocidos y actualmente utilizados se basan en este tipo de enfoque. Entre ellos, quizás los más representativos y ampliamente empleados son aquellos basados en la **generación de Árboles de Decisión simples (DT)**. La razón de su uso radica en su **fácil interpretación**, así como en su **rapidez de cálculo y entrenamiento**. A continuación, se presentan los dos enfoques más conocidos en este sentido: ***Random Forest*** y ***XGBoost***.

### Random Forest

El algoritmo **Random Forest**, propuesto por **Breiman y Cutler** en 2006 (a partir de una idea previa de **Ho** en 1995, conocida como *Random Subspaces*), constituye uno de los ejemplos más representativos del aprendizaje mediante *ensembles*. Combina múltiples clasificadores simples —en este caso, **Árboles de Decisión (DTs)**— en un modelo único y más robusto. Cada árbol del bosque se entrena sobre una **muestra *bootstrap*** (un subconjunto aleatorio con reemplazo) del conjunto de datos original, siguiendo un enfoque de *bagging*. Dado que cada árbol se entrena de manera independiente, el proceso puede ser completamente **paralelizado**.  

En problemas de **clasificación**, la predicción final se obtiene mediante una **votación mayoritaria** entre todos los árboles; en **regresión**, mediante el **promedio** de sus salidas.

Los **Random Forests** destacan por ofrecer un rendimiento notable con **mínimo ajuste de hiperparámetros**. Normalmente, el parámetro más importante es el **número de árboles** (`n_trees` en MLJ o `n_estimators` en *scikit-learn*), que controla el tamaño del *ensemble*. Una heurística común sugiere utilizar:

- *$\sqrt{\textrm{\#features}}$* para tareas de clasificación  
- *$\frac{\textrm{\#features}}{3}$* para tareas de regresión  

Aunque aumentar el número de árboles suele mejorar el rendimiento, este incremento tiende a **saturarse** más allá de los **500–1000 árboles** en la mayoría de los casos prácticos.

Además del proceso de *bootstrapping*, los Random Forest introducen un **segundo nivel de aleatoriedad**: en cada división de nodo, solo se considera un **subconjunto aleatorio de características** como candidatas para la partición. Esto incrementa la **diversidad entre los árboles** y contribuye a **reducir la varianza del modelo**, manteniendo al mismo tiempo una alta capacidad predictiva.  

Un subproducto relevante de este mecanismo es la posibilidad de **cuantificar la importancia de las características**. Analizando cuánto contribuye cada variable a la reducción de la impureza de los nodos a lo largo de todos los árboles, los Random Forest pueden estimar la **importancia relativa de cada característica**. Esta importancia basada en la impureza se emplea frecuentemente para **selección de características**.  
La métrica de impureza más común es el **índice de Gini**, definido como:

$$
G = \sum_{i=1}^C p(i) \cdot (1 - p(i))
$$

donde $C$ es el número de clases y $p(i)$ es la probabilidad de seleccionar aleatoriamente una instancia de la clase $i$. Intuitivamente, mide la probabilidad de clasificar incorrectamente una instancia elegida al azar si las etiquetas se asignaran de acuerdo con la distribución de clases. Para una explicación visual excelente, véase [esta referencia](https://victorzhou.com/blog/gini-impurity/).




In [ ]:
using MLJ
using MLJBase: accuracy
using Plots

# Cargar el modelo Random Forest nativo de Julia
RandomForestClassifier = @load RandomForestClassifier pkg=DecisionTree

# Definir el modelo
models["RF"] = RandomForestClassifier(
    n_trees=8,              
    max_depth=-1,           
    min_samples_split=2,
    n_subfeatures=-1,       
    sampling_fraction=1.0   
)

# Entrenar el modelo
machines["RF"] = machine(models["RF"], train_input, train_output) |> fit!

    
# Evaluar los modelos e imprimir el accuracy
for (name, machine) in machines
    acc = MLJ.accuracy(MLJ.mode.(predict(machine, test_input)), test_output)
    println("$name: $(acc*100) %")
end


### Hiperparámetros clave

| **Parámetro**         | **Descripción** |
|------------------------|-----------------|
| `n_trees`              | Número de árboles en el bosque (equivalente a `n_estimators` en *scikit-learn*). |
| `max_depth`            | Profundidad máxima de cada árbol. Usar `-1` para no establecer límite. |
| `min_samples_split`    | Número mínimo de muestras requeridas para dividir un nodo. |
| `n_subfeatures`        | Número de características aleatorias consideradas en cada división (`-1` utiliza √(#features)). |
| `sampling_fraction`    | Fracción de muestras de entrenamiento empleadas para construir cada árbol (*bootstrapping*). |
| `rng`                  | Generador de números aleatorios para garantizar la reproducibilidad. |

En este ejemplo, el número de árboles (`n_trees`) se define siguiendo la heurística de $\sqrt{\textrm{\#features}}$. Dado que el conjunto de datos utilizado en este ejemplo es relativamente pequeño, los resultados pueden **variar ligeramente entre ejecuciones**, dependiendo de las particiones aleatorias utilizadas durante el entrenamiento.

#### Importancia de características

Una vez entrenado el modelo, puede calcularse la **importancia de las características** a partir de la **reducción media de la impureza de Gini** en todos los árboles del bosque.


In [ ]:
# Extraer los parámetros de los modelos ajustados
fitted_model = fitted_params(machines["RF"])

# Obtner la importancia de las características
feature_importances = fitted_model.features_importance

# Imprimir la importancia de las características
p = bar(
    1:length(feature_importances),
    feature_importances,
    orientation = :horizontal,
    legend = false
)
xlabel!(p, "Gini Gain")
ylabel!(p, "Feature")
title!(p, "Feature Importance")

Como se muestra en la gráfica, gran parte de la información predictiva puede concentrarse en un **número reducido de características**.  
Por tanto, esta métrica también puede emplearse para la **filtración o selección de características**, aspecto que se abordará en secciones posteriores.

Los **Random Forests** representan uno de los métodos *ensemble* más **robustos y ampliamente utilizados**.  
Aprovechan el *bagging* y la **aleatoriedad en la selección de características** para construir árboles diversos, reduciendo la varianza y mejorando la capacidad de generalización.  
Además, proporcionan de forma natural medidas interpretables, como la **importancia de las características**, lo que los convierte no solo en **predictores potentes**, sino también en **herramientas útiles para el análisis exploratorio de datos**.

---

### XGBoost (*eXtreme Gradient Boosting*)

Por último, en esta sección final, debe mencionarse nuevamente el **Gradient Boosting**, en concreto una implementación que en los últimos años se ha hecho muy conocida por su **versatilidad y velocidad**. Esta implementación es conocida como ***XGBoost (eXtreme Gradient Boosting)***, y ha destacado especialmente en competiciones de plataformas como **Kaggle**, debido a su **rapidez de ejecución** y **robustez en los resultados**.  

El ***XGBoost*** constituye un *ensemble* similar al **Random Forest**, pero utiliza un clasificador base diferente denominado **CART** (*Classification and Regression Trees*), en lugar de los simples *Decision Trees*.  
Este cambio responde a la necesidad del algoritmo de obtener **probabilidades asociadas a las decisiones**, como sucede en el *Gradient Tree Boosting*. La otra diferencia fundamental de este algoritmo, al basarse en *Gradient Tree Boosting*, es el **cambio de la estrategia de *bagging* a *boosting*** para la creación de los conjuntos de entrenamiento de los clasificadores.

Posteriormente, esta técnica lleva a cabo un **entrenamiento aditivo**, cuyos pesos se ajustan mediante un **descenso por gradiente** sobre una función de pérdida (*loss function*) previamente definida. Al añadir un término de **regularización** a dicha función de pérdida, puede calcularse la **segunda derivada** de las funciones, lo que permite actualizar los pesos de clasificación de los distintos árboles. El cálculo de este gradiente posibilita ajustar los valores de los clasificadores generados posteriormente, de modo que los pesos concentren la atención en los **patrones clasificados de forma incorrecta**. Los detalles matemáticos de la implementación pueden consultarse en este [enlace](https://xgboost.readthedocs.io/en/stable/tutorials/model.html).

A diferencia de los otros enfoques vistos, **`xgboost` no está actualmente implementado dentro de `scikit-learn`** pero si en **MLJ**. Aun así en este caso **instalar la versión de referencia** si aún no se encuentra disponible en el sistema.

In [ ]:
using Pkg;
Pkg.add("XGBoost")

La API nativa de XGBoost espera matrices numéricas y etiquetas numéricas para clasificación binaria. Como el resto del notebook utiliza etiquetas categóricas de MLJ, crearemos variables específicas para XGBoost sin sobrescribir las variables comunes de entrenamiento y prueba.


In [ ]:
using XGBoost

# Mantener la misma partición, pero con variables específicas para XGBoost.
xgb_train_input = Matrix(train_input)
xgb_test_input = Matrix(test_input)

# XGBoost nativo espera etiquetas numéricas; true (Mina) se codifica como 1.
positive_class = true
xgb_train_output = Int.(train_output .== positive_class)
xgb_test_output = Int.(test_output .== positive_class)

@assert xgb_train_input isa Matrix
@assert xgb_test_input isa Matrix
@assert xgb_train_output isa Vector{Int}
@assert xgb_test_output isa Vector{Int}
@assert size(xgb_train_input, 1) == length(xgb_train_output)
@assert size(xgb_test_input, 1) == length(xgb_test_output)

Una vez realizada esta adaptación de los datos, se puede proceder al **entrenamiento de un modelo** utilizando la biblioteca `xgboost`.  
Para ello, basta con llamar a la función `train` con los parámetros correspondientes.  
Entre estos parámetros, los más relevantes son:

- **eta**: término que determina la **tasa de aprendizaje** o compresión de los pesos después de cada nueva etapa de *boosting*.  
  Toma valores entre 0 y 1.  
- **max_depth**: profundidad máxima de los árboles.  
  Su valor por defecto es 6; incrementarlo permite construir modelos más complejos.  
- **gamma**: parámetro que controla la **reducción mínima de la función de pérdida** necesaria para realizar una nueva partición en una hoja del árbol.  
  Cuanto mayor sea su valor, más **conservador** será el modelo.  
- **alpha** y **lambda**: parámetros que controlan la **regularización L1 y L2**, respectivamente.  
- **objective**: define la **función de pérdida** a utilizar, que puede ser una de las predefinidas en la biblioteca.  
  La lista completa de opciones puede consultarse en este [enlace](https://xgboost.readthedocs.io/en/stable/parameter.html#parameters-for-tree-booster).

Además, solo es necesario establecer el **número máximo de iteraciones** del proceso de *boosting*, como se muestra en el siguiente ejemplo, con **20 rondas** de entrenamiento.

In [ ]:
xgb_data = DMatrix(xgb_train_input, label=xgb_train_output)

xgb_model = xgboost(xgb_data, rounds=20, eta=1, max_depth=6)

En el siguiente fragmento de código, varios parámetros se pasan en forma de **diccionario**, y se calculan **dos métricas diferentes**.  La primera, *error*, se refiere al número de instancias clasificadas incorrectamente sobre el total; la segunda es el **Área Bajo la Curva ROC (AUC)**.

### Pregunta
> ❓ ¿Cuál es el nombre canónico de la primera medida que se está monitorizando?

In [ ]:
param = ["max_depth" => 2,
         "eta" => 1,
         "objective" => "binary:logistic"]
metrics = ["error", "auc"]
xgb_model = xgboost(DMatrix(xgb_train_input, label=xgb_train_output), rounds=20, param=param, metrics=metrics)

pred = predict(xgb_model, xgb_train_input)

***Importante***

En caso de utilizar un **conjunto de validación**, este debe pasarse mediante el parámetro **`evals`** de la función de entrenamiento.  
Además, **solo cuando dicho parámetro `evals` está definido**, es posible establecer las **rondas de parada temprana** mediante el parámetro **`early_stopping_rounds`** de la función `train`.  

El código sería similar al siguiente:

```julia
evals = DMatrix(val_input, label=val_output)
xgb_model = xgb.train(param, xgb_train_input, num_round, label=xgb_train_output, 
                      evals=evals, early_stopping_rounds=10)
```
El valor proporcionado en la salida corresponde a la **suma de las salidas de los árboles**, situándose **entre 0 y 1** para representar la **probabilidad de pertenencia a una clase dada**. Dado que se trata de un problema de clasificación binaria, basta con establecer un umbral de 0.5 sobre la salida para determinar la clase predicha.

In [ ]:
using Statistics: mean
using XGBoost: predict as predict_xgb

pred = predict_xgb(xgb_model, xgb_test_input)
print("Error of XGBoost = ", mean((pred .> 0.5) .!= xgb_test_output), "\n")

Finalmente, al igual que en el caso de **Random Forest**, es posible **identificar la importancia de las características** y representarla gráficamente según su posición en el **ranking de relevancia**. Con el siguiente fragmento de código puede visualizarse dicho indicador, **ordenado de forma ascendente**:


In [ ]:
feature_gain = [(first(x), last(x)) for x in importance(xgb_model)]
feature, gain = first.(feature_gain), last.(feature_gain)

using Plots;

p = bar(feature, y=gain, orientation="h", legend=false)
xlabel!(p,"Gain")
ylabel!(p,"Feature")
title!("Feature Importance")

Como puede observarse, **no todas las características tienen la misma importancia**.  
Debe tenerse en cuenta que el eje *Feature* identifica la **posición en el vector de características**, el cual se ordena **por defecto según el valor de ganancia (*gain*)**.

## Técnicas más modernas

En los últimos años han surgido nuevas implementaciones de algoritmos basados en *ensemble learning* que buscan **optimizar el rendimiento y la eficiencia computacional** de los métodos clásicos de *Gradient Boosting*.  
Entre estas destacan **LightGBM** y **CatBoost**, dos librerías de alto rendimiento ampliamente utilizadas tanto en la investigación como en la práctica profesional.

Ambos algoritmos parten de la misma base teórica del *Gradient Boosting*, pero introducen **mejoras sustanciales en velocidad, manejo de datos categóricos y escalabilidad**.  
Estas optimizaciones los convierten en herramientas especialmente adecuadas para conjuntos de datos **grandes, complejos o heterogéneos**.  

En las siguientes secciones se presentan brevemente las características principales de **LightGBM** y **CatBoost**, junto con ejemplos opcionales de uso. La ruta recomendada en este tutorial es emplear los *wrappers* de **MLJ** cuando estén disponibles. Si estos fallan o no están instalados en el entorno, una alternativa práctica es acceder a las bibliotecas Python mediante `PyCall` y `pyimport`.

> **Nota docente:** el conjunto Sonar es pequeño y contiene variables numéricas. LightGBM y CatBoost son especialmente útiles en problemas tabulares grandes, con muchas características o con variables categóricas. En este tutorial se incluyen como técnicas avanzadas, no como celdas obligatorias para ejecutar el flujo principal.

#### LightGBM

**LightGBM (Light Gradient Boosting Machine)** es una implementación moderna y altamente optimizada del algoritmo de *Gradient Boosting*, desarrollada por **Microsoft Research**.  
Está diseñada específicamente para ofrecer **gran velocidad y eficiencia** en contextos con **grandes volúmenes de datos** y **alto número de características**, resolviendo algunos de los problemas de escalabilidad presentes en **XGBoost**.

A diferencia de otros algoritmos de *boosting*, LightGBM introduce dos innovaciones fundamentales:

1. **Crecimiento de los árboles orientado a hojas (Leaf-wise growth)**:  
   En lugar de crecer los árboles nivel por nivel (*level-wise*), como hace XGBoost, LightGBM expande las ramas que más reducen la pérdida (*loss*).  
   Esto produce modelos más precisos, aunque con mayor riesgo de sobreajuste si no se controla la profundidad.

2. **Histogram-based Decision Tree Learning**:  
   Los valores continuos se agrupan en histogramas, reduciendo el uso de memoria y acelerando el cálculo de los puntos de división óptimos.

#### Uso recomendado y alternativas

En Julia se puede intentar primero la integración con MLJ mediante `MLJLightGBMInterface`, lo que permite mantener el mismo flujo de `machine`, `fit!` y `evaluate!` usado en el resto del notebook. Si el *wrapper* no funciona en el entorno disponible, puede usarse la implementación Python mediante `PyCall.pyimport("lightgbm")`.

Los parámetros más habituales son `num_iterations`, `learning_rate`, `num_leaves`, `max_depth`, `feature_fraction`, `bagging_fraction` y las métricas de evaluación como `auc` o `binary_logloss`.

#### Principales parámetros de LightGBM

* `objective`: función objetivo que se optimiza. En clasificación binaria suele usarse `"binary"`; en multiclase se emplean objetivos específicos para varias clases.

* `num_iterations` o `n_estimators`: número de iteraciones de *boosting*, equivalente al número de árboles añadidos secuencialmente. Valores altos pueden mejorar el ajuste, pero también aumentar el sobreajuste si no se controla la complejidad.

* `learning_rate`: tasa de aprendizaje. Controla cuánto contribuye cada nuevo árbol al modelo final. Valores pequeños suelen requerir más iteraciones, pero pueden generalizar mejor.

* `num_leaves`: número máximo de hojas por árbol. Es uno de los parámetros más importantes en LightGBM, porque su crecimiento orientado a hojas puede generar árboles muy complejos. Valores altos aumentan la capacidad del modelo y el riesgo de sobreajuste.

* `max_depth`: profundidad máxima del árbol. Permite limitar explícitamente la complejidad, especialmente cuando `num_leaves` es alto.

* `feature_fraction`: fracción de características utilizada para construir cada árbol. Ayuda a introducir diversidad y reducir sobreajuste cuando hay muchas variables.

* `bagging_fraction`: fracción de ejemplos usada en cada iteración. Actúa como muestreo de filas y puede mejorar la generalización.

* `bagging_freq`: frecuencia con la que se aplica el muestreo definido por `bagging_fraction`. Si vale 0, normalmente no se aplica *bagging*.

* `early_stopping_round`: número de iteraciones sin mejora en un conjunto de validación antes de detener el entrenamiento. Requiere disponer de datos de validación.

* `metric`: métrica o métricas monitorizadas durante el entrenamiento, por ejemplo `"auc"`, `"binary_logloss"` o `"multi_logloss"`.

In [ ]:
# Ejemplo opcional: instalación de la interfaz MLJ para LightGBM.
# Ejecuta estas líneas solo si quieres probar LightGBM y el entorno no lo tiene instalado.
#
# using Pkg
# Pkg.add("MLJLightGBMInterface")
#
# using MLJ
# LightGBMClassifier = @load LightGBMClassifier pkg=MLJLightGBMInterface
#
# models["LightGBM"] = LightGBMClassifier(
#     num_iterations=100,
#     learning_rate=0.1,
#     max_depth=-1
# )
#
# machines["LightGBM"] = machine(models["LightGBM"], train_input, train_output) |> fit!
# acc = MLJ.accuracy(MLJ.mode.(predict(machines["LightGBM"], test_input)), test_output)
# println("LightGBM: $(acc*100) %")


#### Alternativa mediante PyCall

Si `MLJLightGBMInterface` no está disponible o no carga correctamente, puede utilizarse la biblioteca Python directamente. En ese caso se pierde la integración automática con `machine` y `evaluate!`, por lo que es necesario convertir manualmente los datos a matrices y etiquetas numéricas, como se hizo en la sección de XGBoost.

In [ ]:
# Ejemplo opcional: uso de LightGBM vía PyCall.
# Requiere que el paquete Python lightgbm esté instalado en el Python usado por PyCall.
#
# using PyCall
# using Statistics: mean
#
# Si lightgbm no está instalado en Python, una forma posible, dependiente del entorno, es:
# pyimport("subprocess").check_call(["python", "-m", "pip", "install", "lightgbm"])
#
# lgb = pyimport("lightgbm")
#
# lgb_train_input = Matrix(train_input)
# lgb_test_input = Matrix(test_input)
# lgb_train_output = Int.(train_output .== true)
# lgb_test_output = Int.(test_output .== true)
#
# lgb_model = lgb.LGBMClassifier(
#     objective="binary",
#     n_estimators=100,
#     learning_rate=0.1,
#     num_leaves=31
# )
# lgb_model.fit(lgb_train_input, lgb_train_output)
# lgb_pred = lgb_model.predict(lgb_test_input)
# println("LightGBM error = $(mean(lgb_pred .!= lgb_test_output))")

La ruta MLJ es preferible cuando se desea comparar LightGBM con el resto de modelos del notebook mediante el mismo flujo de trabajo. La ruta `PyCall` es útil cuando el *wrapper* de MLJ no está disponible o cuando se necesita acceder a opciones específicas de la implementación Python.

### CatBoost

**CatBoost (Categorical Boosting)** es un algoritmo de *Gradient Boosting* desarrollado por **Yandex**, diseñado específicamente para tratar de forma eficiente **variables categóricas** sin necesidad de un preprocesamiento intensivo.  
Su principal fortaleza radica en que **codifica internamente las variables categóricas** mediante combinaciones estadísticas y ordenamientos aleatorios, evitando el uso de *one-hot encoding* y reduciendo el riesgo de sobreajuste.

CatBoost también introduce innovaciones que mejoran la estabilidad del modelo:

1. **Orden aleatorio controlado (*ordered boosting*)**:  
   Se evita la dependencia de la muestra durante el cálculo del gradiente, reduciendo el *prediction shift*.

2. **Combinación eficiente de características categóricas**:  
   Genera automáticamente nuevas variables basadas en combinaciones de categorías, mejorando la capacidad predictiva sin intervención manual.

#### Uso recomendado y alternativas

Como en el caso anterior, la primera opción dentro de este tutorial es intentar la integración con MLJ mediante `MLJCatBoostInterface`. Si el *wrapper* falla, puede recurrirse a la implementación Python con `PyCall.pyimport("catboost")`.

En este conjunto Sonar, CatBoost no aprovecha su principal ventaja, ya que las 60 variables son numéricas. Su interés principal aparece en problemas tabulares con variables categóricas reales.

In [ ]:
# Ejemplo opcional: instalación de la interfaz MLJ para CatBoost.
# Ejecuta estas líneas solo si quieres probar CatBoost y el entorno no lo tiene instalado.
#
# using Pkg
# Pkg.add("MLJCatBoostInterface")
#
# using MLJ
# CatBoostClassifier = @load CatBoostClassifier pkg=MLJCatBoostInterface
#
# models["CatBoost"] = CatBoostClassifier(
#     iterations=500,
#     learning_rate=0.05,
#     depth=8
# )
#
# machines["CatBoost"] = machine(models["CatBoost"], train_input, train_output) |> fit!
# acc = MLJ.accuracy(MLJ.mode.(predict(machines["CatBoost"], test_input)), test_output)
# println("CatBoost: $(acc*100) %")

In [ ]:
# Ejemplo opcional: instalación de PyCall si se quiere usar la biblioteca Python.
#
# using Pkg
# Pkg.add("PyCall")
#
# La biblioteca Python catboost debe estar instalada en el Python usado por PyCall.
# Una forma posible, dependiente del entorno, es:
# using PyCall
# pyimport("subprocess").check_call(["python", "-m", "pip", "install", "catboost"])


El uso mediante `PyCall` requiere preparar manualmente los datos y llamar a la API Python. Este enfoque es menos homogéneo que MLJ, pero resulta útil como alternativa práctica cuando el *wrapper* no funciona.

In [ ]:
# Ejemplo opcional: uso de CatBoost vía PyCall.
# Requiere que el paquete Python catboost esté instalado en el Python usado por PyCall.
#
# using PyCall
# using Statistics: mean
# cb = pyimport("catboost")
#
# cb_train_input = Matrix(train_input)
# cb_test_input = Matrix(test_input)
# cb_train_output = Int.(train_output .== true)
# cb_test_output = Int.(test_output .== true)
#
# cb_model = cb.CatBoostClassifier(
#     iterations=500,
#     learning_rate=0.05,
#     depth=8,
#     verbose=false
# )
# cb_model.fit(cb_train_input, cb_train_output)
# cb_pred = cb_model.predict(cb_test_input)
# println("CatBoost error = $(mean(vec(cb_pred) .!= cb_test_output))")

En ambos casos, si se usa la ruta Python, hay que recordar que las predicciones pueden devolverse con una forma distinta a la de Julia; por eso en el ejemplo se usa `vec(cb_pred)` antes de comparar con las etiquetas esperadas.

In [ ]:
# Resumen de decisión práctica:
#
# 1. Usa MLJ si quieres mantener el mismo flujo que el resto del tutorial.
# 2. Usa PyCall si el wrapper MLJ no está disponible o necesitas opciones específicas de Python.
# 3. Evita comparar estos modelos con hold-out único en datasets pequeños; usa validación cruzada.

#### Principales parámetros de CatBoost

* `iterations`: número de iteraciones de *boosting*, equivalente al número de árboles construidos secuencialmente. Un valor alto puede mejorar el ajuste, pero también incrementar el coste computacional y el riesgo de sobreajuste.

* `learning_rate`: tasa de aprendizaje, típicamente entre 0.01 y 0.1. Valores bajos suelen requerir más `iterations`, pero pueden mejorar la generalización.

* `depth`: profundidad máxima de los árboles. Controla directamente la complejidad de cada árbol y es uno de los parámetros más relevantes para evitar sobreajuste.

* `l2_leaf_reg`: regularización L2 aplicada a los valores de las hojas. Valores mayores penalizan modelos demasiado complejos.

* `eval_metric`: métrica de evaluación utilizada durante el entrenamiento, por ejemplo `AUC`, `Accuracy`, `Logloss` o métricas específicas de multiclase.

* `early_stopping_rounds`: número de iteraciones sin mejora en un conjunto de validación antes de detener el entrenamiento. Es útil para evitar sobreajuste, pero requiere validación explícita.

* `bootstrap_type`: método de muestreo usado durante el entrenamiento, como `Bayesian`, `Bernoulli` o `Poisson`, dependiendo del backend y del dispositivo.

* `cat_features`: índice o lista de variables categóricas. Es especialmente importante en CatBoost, porque permite tratar estas variables internamente sin aplicar manualmente *one-hot encoding*.

* `loss_function`: función de pérdida optimizada, por ejemplo `Logloss` para clasificación binaria o `MultiClass` para clasificación multiclase.

* `task_type`: define si se entrena en CPU o GPU, cuando el entorno y la instalación lo soportan.

#### Consideraciones prácticas

LightGBM y CatBoost no sustituyen automáticamente a Random Forest o XGBoost. En datasets pequeños pueden sobreajustar si se usan demasiadas iteraciones, árboles demasiado profundos o valores altos de `num_leaves`. Su evaluación debería hacerse con validación cruzada y, si se ajustan hiperparámetros, con una partición interna o un esquema de validación anidada.

En resumen: utiliza **MLJ** para integración homogénea con el resto del notebook y **PyCall** como alternativa cuando el *wrapper* no esté disponible o no funcione correctamente en el entorno.




## Notas sobre Julia

### Comprendiendo `coerce` en MLJ

Al trabajar con conjuntos de datos en **MLJ**, es importante entender que el sistema distingue entre los **tipos de máquina** (por ejemplo, `Int64`, `Float64`, `String`) y los **tipos científicos** (*scientific types* o *scitypes*), que describen cómo deben **interpretarse los datos** en el contexto del modelado.

### Por qué se necesita `coerce`

Los modelos de **MLJ** no se basan en los tipos nativos de Julia, sino que esperan que las variables tengan **significados científicos** explícitos:
- Una columna numérica puede representar una característica **continua**.  
- Una columna de texto (*string*) puede representar una característica **categórica**.  
- Una columna booleana o entera puede ser **ordenada** o **no ordenada**, según el contexto.

Dado que Julia no puede inferir esto automáticamente, utilizamos la función `coerce()` para **indicar explícitamente a MLJ cómo debe interpretar cada columna**.  
De este modo, se garantiza la **compatibilidad entre los datos y el modelo de MLJ** que se desea emplear.

---

### Ejemplo



In [ ]:
using MLJ, DataFrames, CSV

# Carrgar el conjunto de datos Sonar
data = CSV.read("sonar.csv", DataFrame)

# Inspeccionar los tipos de columna actuales
schema(data)

# Supongamos que la última columna es el objetivo ('Rock' o 'Mine')
y, X = unpack(data, ==(:Target), rng=123)

# Convertir la variable objetivo a categórica
y = coerce(y, Multiclass)

# Convertir todas las columnas de características a continuas
X = coerce(X, autotype(X, rules = (:discrete_to_continuous,)))

# Verificar los nuevos tipos
schema(X)

### Ejemplos comunes

| Situación                             | Qué hacer                | Ejemplo                               |
| ------------------------------------- | ------------------------ | ------------------------------------- |
| Etiquetas categóricas almacenadas como cadenas | Convertir a `Multiclass` | `y = coerce(y, Multiclass)`           |
| Características numéricas             | Convertir a `Continuous` | `X = coerce(X, :var1 => Continuous)`  |
| Inferencia automática                 | Usar `autotype()`        | `autotype(X)`                         |
| Comprobar los tipos científicos actuales | Usar `schema()`          | `schema(X)`                           |

### Qué ocurre si se omite `coerce`

Si se omite el paso de coerción:

* **MLJ puede interpretar incorrectamente la variable objetivo** como continua, impidiendo el uso de modelos de clasificación.  
* **Algunos algoritmos** (por ejemplo, *DecisionTree*, *NaiveBayes*) pueden **fallar o generar predicciones incorrectas**.  
* La función `machine()` puede **rechazar la vinculación entre el modelo y los datos** si los tipos no son compatibles.


## Voting Classifier

Siguiendo estas líneas, se presentan varios ejemplos de cómo utilizar el nuevo **Voting Classifier** que hemos implementado.

En ejemplos previos, construimos el *ensemble* de la siguiente manera:

```julia
voting_hard = VotingClassifier(models=base_models, voting=:hard)
voting_soft = VotingClassifier(models=base_models, voting=:soft)

mach_hard = machine(voting_hard, train_input, train_output) |> fit!
mach_soft = machine(voting_soft, train_input, train_output) |> fit!
```

* `voting=:hard` → utiliza votación mayoritaria (basada en las etiquetas de clase).  

* `voting=:soft` → utiliza votación por promedio de probabilidades (requiere modelos que generen probabilidades como salida).  

Los siguientes ejemplos muestran cómo **modificar e integrar dinámicamente** el `VotingClassifier` dentro de tu flujo de trabajo.


In [ ]:
### Ejemplo 1: Cambio dinámico del tipo de votación
voting_hard = VotingClassifier(models=base_models, voting=:hard)
voting_soft = VotingClassifier(models=base_models, voting=:soft)

ensemble = deepcopy(voting_hard)
println("\nTipo de votación actual: $(ensemble.voting)")

ensemble.voting = :soft
println("Tipo de votación cambiado a: $(ensemble.voting)")


In [ ]:
# Ejemplo 2: Uso del `VotingClassifier` en una *Pipeline*
pipe_hard = @pipeline(
    Standardizer(),
    VotingClassifier(models=base_models, voting=:hard)
)

pipe_soft = @pipeline(
    Standardizer(),
    VotingClassifier(models=base_models, voting=:soft)
)

In [ ]:
# Ejemplo 3: Validación cruzada comparando ambas estrategias de votación
using Statistics: mean

voting_hard = VotingClassifier(models=base_models, voting=:hard)
voting_soft = VotingClassifier(models=base_models, voting=:soft)

cv_hard = evaluate!(
    machine(voting_hard, train_input, train_output),
    resampling=CV(nfolds=5),
    measure=accuracy
)
println("Hard Voting CV accuracy: $(round(mean(cv_hard.measurement)*100, digits=2)) %")

cv_soft = evaluate!(
    machine(voting_soft, train_input, train_output),
    resampling=CV(nfolds=5),
    measure=accuracy
)
println("Soft Voting CV accuracy: $(round(mean(cv_soft.measurement)*100, digits=2)) %")